In [508]:
import numpy as np
import pandas as pd
import random
# load et simulation function from simulate_population.py: 
from simulate_population import sim_population

## 1 Simulate data

In [531]:
pop = sim_population(N=20000, step_forward=2, randomseed=42)
pop.step() # move +2y
pop.step() # move + 2y
pop.step() #move + 2y
pop.step() #move + 2y
pop.step() #move + 2y

In [532]:
pop.history[1].head()

,id,start,end,age_start,bmi,hyp,smoke,sex,eth,first_a,...,time_a,time_b,time_c,time_d,time_e,event_a,event_b,event_c,event_d,event_e
0,1,2,4,62.1,-1.8,0,0,1,0,NaN,...,2.080,0.264,1.118,55.531,52.353,0,1,1,0,0
1,2,2,4,43.0,-0.4,0,0,1,2,2.172,...,0.172,2.035,17.578,47.791,19.104,1,0,0,0,0
2,3,2,4,66.9,-1.7,0,0,0,0,2.569,...,0.569,0.153,0.012,48.803,65.261,1,1,1,0,0
3,4,2,4,57.7,2.0,1,0,0,2,0.100,...,0.178,0.342,0.303,51.891,272.711,1,1,1,0,0
4,5,2,4,23.4,0.2,0,0,1,0,NaN,...,27.104,4.313,43.508,2.099,7.984,0,0,0,0,0


In [511]:
#a single long dataframe - if needed
#long_df = pd.concat(pop.history, ignore_index=True)
#long_df = long_df.sort_values(by=['id', "start"])
#long_df.head(10)

### b) Create df_long 
	id	age_start	bmi	hyp	smoke	sex	eth	event	time	age
	2	43.0	-0.4	0	0	1	2	first_a	2.172	45.172
	3	66.9	-1.7	0	0	0	0	first_a	2.569	69.469

In [535]:
event_cols = ["first_a", "first_b", "first_c", "first_d", "first_e"]
context_cols = ['age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth']
df_short = pop.history[5][["id"]+context_cols + event_cols].copy()
df_long = df_short.melt(id_vars = ["id"] + context_cols, 
                       value_vars = event_cols, 
                       var_name = "event", 
                       value_name = "time").dropna(subset=["time"])
df_long["age"] = df_long["age_start"] + df_long["time"]
df_long = df_long.sort_values(by=['id', "age"])
df_long.head()

,id,age_start,bmi,hyp,smoke,sex,eth,event,time,age
20000,1,62.1,-1.8,0,0,1,0,first_b,0.059,62.159
40000,1,62.1,-1.8,0,0,1,0,first_c,3.118,65.218
1,2,43.0,-0.4,0,0,1,2,first_a,2.172,45.172
20001,2,43.0,-0.4,0,0,1,2,first_b,8.772,51.772
40002,3,66.9,-1.7,0,0,0,0,first_c,0.860,67.760


In [ ]:
df_short.head()

## 2 Check CoxPH

* eth_beta = np.array([0 if (x==0) else 0.5 if (x==1) else 2 for x in df["eth"]])
* chances are higher if there was a heart failure before 
* eventb_beta = (~df["first_b"].isna()).astype(int) * 0.3
* lp1 = 0.1*np.exp(0.5*df.bmi + 0.7*df.hyp + 0.4*(df.age-50)/15 + eth_beta + eventb_beta)
* time_a = 0.01 + np.round(rng.exponential(1/lp1,len(df)),3)

In [513]:
import pandas as pd
from lifelines import CoxPHFitter
pop = sim_population(N=10000, step_forward=5, randomseed=42)
df_cox = pop.history[0][['end', 'event_b', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']].copy()

In [514]:
cph = CoxPHFitter()
cph.fit(df_cox, duration_col='end', event_col='event_b')
s =cph.summary[['coef', 'se(coef)', 'p']]
round(s,4)

,coef,se(coef),p
covariate,,,
age,0.0267,0.0009,0.0000
bmi,0.0356,0.0152,0.0193
hyp,0.4815,0.0318,0.0000
smoke,-0.0112,0.0388,0.7718
sex,0.0256,0.0278,0.3577
eth,-0.0104,0.0208,0.6163


In [515]:
df1 = pop.history[2][['start','end', 'event_d', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']]
cph.fit(df1, duration_col='end', event_col='event_d', entry_col="start")
s1 =cph.summary[['coef', 'se(coef)', 'p']]
round(s1,4)

IndexError: list index out of range

In [ ]:
# Now check if going from 0 till the end and taking ever happening event_a returns similar coefficients: 
df1_first = pop.history[2][['first_e','end', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']].copy()
df1_first['event'] = df1_first['first_e'].notna().astype(int) # if a ever happened = 1
df1_first['time'] = df1_first['first_e'].where(df1_first['first_e'].notna(), 10) #time is from first_a column
cph.fit(df1_first[['time','event', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']], 
        duration_col='time', event_col='event')
s1_first = cph.summary[['coef', 'se(coef)', 'p']]
round(s1_first,4)

In [ ]:
df_short["event_a_1"] =  df_short['first_a'].notna().astype(int)
df_short['time'] = df_short['first_a'].where(df_short['first_a'].notna(), 12)
cph.fit(df_short[['time','event_a_1', 'age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth']], 
        duration_col='time', event_col='event_a_1')
sdfg_first = cph.summary[['coef', 'se(coef)', 'p']]
round(sdfg_first,4)

## 3 Define transformer

Embeddings: 1) Static context (time-invariant)   sex, eth; 2) Dynamic context (time-varying, per step)  age, bmi, hyp, smoke, 3) Token per step  {NONE, A, B, C, D, E}

Encodings (to sum together): 1) Event token embedding; 2) Dynamic covariate embedding; 3) Static covariate embedding (broadcast over time)

#### a) Shape df_long for transformer

In [516]:
def build_event_sequences(df_long):
    sequences = []
    for pid, g in df_long.groupby("id"):
        g = g.sort_values("time")
        tokens = g["event"].str.replace("first_", "", regex=False).tolist()
        features = g[["age", "bmi", "hyp", "smoke", "sex", "eth"]].to_numpy()
        times = g["time"].to_numpy()
        sequences.append({ "id": pid, "tokens": tokens, "features": features,"times": times })
    return sequences

In [518]:
def pad_sequences_mixed(sequences, vocab):
    token_seqs = []
    dt_seqs = []
    bmi_seqs = []
    cov_invar_seqs = []
    lengths = []

    for s in sequences:
        tokens = [vocab[t] for t in s["tokens"]]
        dt = np.diff(np.concatenate([[0.0], s["times"]]))[:, None]
        bmi = s["features"][:, 1:2]  # assuming BMI is second column
        cov_invar = s["features"][0, [0,2,3,4,5]]  # age, hyp, smoke, sex, eth
        token_seqs.append(tokens)
        dt_seqs.append(dt)
        bmi_seqs.append(bmi)
        cov_invar_seqs.append(cov_invar)
        lengths.append(len(tokens))
    
    max_len = max(lengths)
    
    def pad(x, value=0):
        return np.array([xi + [value]*(max_len - len(xi)) for xi in x])
    
    X_tokens = torch.tensor(pad(token_seqs), dtype=torch.long)
    X_dt = torch.tensor(
        np.stack([np.vstack([d, np.zeros((max_len - len(d), 1))]) for d in dt_seqs]),
        dtype=torch.float32
    )
    X_bmi = torch.tensor(
        np.stack([np.vstack([b, np.zeros((max_len - len(b), 1))]) for b in bmi_seqs]),
        dtype=torch.float32
    )
    X_cov_invar = torch.tensor(np.stack(cov_invar_seqs), dtype=torch.float32)
    lengths = torch.tensor(lengths)
    
    return X_tokens, X_dt, X_bmi, X_cov_invar, lengths



#### b) Transformer 
* next-event prediction/Delphi2M style
* P(event_k | event_<k, covariates_<k)

In [519]:
class EventSequenceTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model) #event tokens
        self.time_emb = nn.Linear(1, d_model)   # dt
        self.cov_invar_emb = nn.Linear(5, d_model)    # age, hyp, smoke, sex, eth
        self.cov_time_emb = nn.Linear(1, d_model)       #bmi, time-variant covariate
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True )
        
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.out = nn.Linear(d_model, vocab_size)
    
    def forward(self, token_ids, dt, cov_time, cov_invar):
        """ 
        token_ids: [B, T] event tokens
        dt: [B, T, 1] delta-times
        cov_time: [B, T, 1] time-varying covariates (BMI)
        cov_invar: [B, n_timeinv] baseline covariates
        """
        x = self.token_emb(token_ids) + self.time_emb(dt) + self.cov_time_emb(cov_time)
        
        # embed time-invariant covariates and broadcast
        cov_invar_emb = self.cov_invar_emb(cov_invar).unsqueeze(1)  # [B,1,d]
        
        x = x + cov_invar_emb  # broadcast along sequence
        
        x = self.encoder(x)
        logits = self.out(x)
        
        return logits

#### c) Loss

In [520]:
#STANDARD CROSS_ENTROPY LOSS

def masked_cross_entropy(logits, targets, lengths):
    """     logits:  [B, T-1, V]     targets: [B, T-1]     lengths: [B]   """
    B, T, V = logits.shape
    mask = ( torch.arange(T)[None, :] < (lengths - 1)[:, None])
    loss = torch.nn.functional.cross_entropy(
        logits.reshape(-1, V), targets.reshape(-1),  reduction="none" )
    loss = loss.view(B, T)
    
    return (loss * mask).sum() / mask.sum()

In [521]:
# ALTERNATIVE DELPHI-2M style loss

def masked_competing_exp_loss(logits, targets, dt, lengths, alpha=1.0):
    """  logits:[B, T, K], targets: [B, T], dt: [B, T], lengths: [B] """
    B, T, K = logits.shape
    device = logits.device
    targets = targets.long()
    mask = (torch.arange(T, device=device)[None, :]< lengths[:, None])
    # event loss
    loss_event = torch.nn.functional.cross_entropy(
        logits.reshape(-1, K), targets.reshape(-1), reduction="none" ).view(B, T)
    # time loss
    log_lambda_sum = torch.logsumexp(logits, dim=-1)
    lambda_sum = torch.exp(log_lambda_sum)
    loss_time = -log_lambda_sum + lambda_sum * dt
    loss = loss_event + alpha * loss_time
    return (loss * mask).sum() / mask.sum()

#### d) Training function

In [522]:
# TRAINING FUNCTION 

def train_transformer(model, sequences, vocab, n_epochs=10,
                       lr=1e-3, batch_size=32, alpha = 1.0, device="cpu" ):
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # build tensors once
    X_tokens, X_dt, X_bmi, X_cov_invar, lengths = pad_sequences_mixed(sequences, vocab)

    N = X_tokens.size(0)
    for epoch in range(n_epochs):
        perm = torch.randperm(N)
        total_loss = 0.0
        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            tokens = X_tokens[idx]
            dt = X_dt[idx]
            bmi = X_bmi[idx]
            cov_invar = X_cov_invar[idx]
            lens = lengths[idx]
            optimizer.zero_grad()
            # predict hazards for the next event
            logits = model(tokens[:, :-1],  dt[:, :-1], bmi[:, :-1], cov_invar ) #history, covariates then, previous gap
             # [B, T-1, K]

            # compute the loss
            loss = masked_competing_exp_loss(
                logits=logits, 
                targets=tokens[:, 1:],   # next event
                dt=dt[:, 1:].squeeze,             # waiting time to next event
                lengths=lens - 1,
                alpha=alpha
            )
            ## alternative (no hazard, just cross-entropy):
            ## loss = masked_cross_entropy( logits, tokens[:, 1:], lens)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        avg_loss = total_loss / N
        print(f"Epoch {epoch+1:03d} | loss = {avg_loss:.4f}")


## 4 Fit transformer to data

In [523]:
#get sequences from df_long
dfdf = build_event_sequences(df_long)

In [ ]:
vocab = {"a": 0, "b": 1, "c": 2, "d": 3, "e": 4}
model = EventSequenceTransformer(vocab_size=len(vocab), d_model=64,  nhead=4)
train_transformer(model, dfdf, vocab, n_epochs=20, lr=1e-3, batch_size=64)

## 4 Test transformer on df_long_test

#### a) create test set

In [ ]:
# create test set
pop_test = sim_population(N=20000, step_forward=2, randomseed=3456)
pop_test.step() # move +2y
pop_test.step() # move + 2y
pop_test.step() #move + 2y
pop_test.step() #move + 2y
pop_test.step() #move + 2y
event_cols = ["first_a", "first_b", "first_c", "first_d", "first_e"]
context_cols = ['age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth']
df_long_test = pop_test.history[5][["id"]+context_cols + event_cols].melt(id_vars = ["id"] + context_cols, 
                       value_vars = event_cols, 
                       var_name = "event", 
                       value_name = "time").dropna(subset=["time"])
df_long_test["age"] = df_long_test["age_start"] + df_long_test["time"]
df_long_test = df_long_test.sort_values(by=['id', "age"])
del pop_test
df_long_test.head()

#### b) predicting for df_long_test

dynamic risk scores directly from your df_long_test, given the history up to some status_time for each patient

In [ ]:
# make sequences from df_long_test for transformer learning
sequences_test = build_event_sequences(df_long_test)
X_tokens_test, X_feats_test, X_dt_test, lengths_test = pad_sequences(sequences_test, vocab)

In [ ]:
# probability event k happens within horizon tau (cum_haz from logit over time tau)
# P(T ≤ τ, event = k) = (λ_k / Λ) * (1 − exp(-Λ τ))
def cumulative_incidence(logits, tau):
    lambdas = torch.exp(logits)                # [B, T, K]
    lambda_sum = lambdas.sum(dim=-1, keepdim=True)
    return (lambdas / lambda_sum) * (1 - torch.exp(-lambda_sum * tau))

In [ ]:
# make predictions given the history: 
def build_test_sequences(df_long, status_time = 5):
    df_long["status_time_col"] = status_time
    sequences = []
    for pid, g in df_long.groupby("id"):
        g = g.sort_values("time")
        cutoff = g["status_time_col"].iloc[0] if "status_time_col" in g else g["time"].max()
        g_cut = g[g["time"] <= cutoff]
        if len(g_cut) == 0:
            continue
        tokens = g_cut["event"].str.replace("first_", "", regex=False).tolist()
        features = g_cut[["age", "bmi", "hyp", "smoke", "sex", "eth"]].to_numpy()
        times = g_cut["time"].to_numpy()
        sequences.append({"id": pid, "tokens": tokens, "features": features, "times": times})
    return sequences

function that takes your trained model, a test dataframe, and a status_time column, and outputs per-event risk scores and Harrell’s C-index. It will handle truncating sequences, padding, predicting hazards, aggregating into risk scores, and computing the C-index.

In [ ]:
import torch
import numpy as np
from lifelines.utils import concordance_index

def compute_dynamic_cindex(model, df_test, vocab, status_time_col="status_time", device="cpu"):
    """
    Compute dynamic Harrell's C-index for each event type given history up to status_time.

    Parameters
    ----------
    model : nn.Module
        Trained EventSequenceTransformer
    df_test : pd.DataFrame
        Long-format test data with columns: id, time, event, covariates
    vocab : dict
        Mapping of event names to integer IDs
    status_time_col : str
        Column name in df_test giving the evaluation time (history cutoff)
    device : str
        "cpu" or "cuda"

    Returns
    -------
    c_index_dict : dict
        Harrell's C-index per event
    risk_scores : np.ndarray
        Risk scores [n_patients, n_events]
    """
    model.eval()
    
    # 1. Build sequences truncated at status_time
    sequences = []
    for pid, g in df_test.groupby("id"):
        g = g.sort_values("time")
        cutoff = g[status_time_col].iloc[0] if status_time_col in g else g["time"].max()
        g_cut = g[g["time"] <= cutoff]
        if len(g_cut) == 0:
            continue
        tokens = g_cut["event"].str.replace("first_", "", regex=False).tolist()
        features = g_cut[["age", "bmi", "hyp", "smoke", "sex", "eth"]].to_numpy()
        times = g_cut["time"].to_numpy()
        sequences.append({"id": pid, "tokens": tokens, "features": features, "times": times})
    
    if len(sequences) == 0:
        raise ValueError("No sequences found up to the given status_time.")

    # 2. Pad sequences
    X_tokens, X_feats, X_dt, lengths = pad_sequences(sequences, vocab)
    X_tokens = X_tokens.to(device)
    X_feats = X_feats.to(device)
    X_dt = X_dt.to(device)
    lengths = lengths.to(device)
    
    # 3. Predict hazards
    with torch.no_grad():
        logits = model(
            X_tokens[:, :-1],
            X_feats[:, :-1],
            X_dt[:, :-1]
        )
        hazards = torch.softmax(logits, dim=-1)  # [B, T-1, K]
    
    # 4. Aggregate into risk scores
    risk_scores = hazards.sum(dim=1).cpu().numpy()  # sum over time -> [B, K]
    
    # 5. Prepare true event indicators and times for each event
    event_types = list(vocab.keys())
    true_events = []
    true_times = []
    
    for s in sequences:
        last_time = s["times"][-1]
        for k, ev in enumerate(event_types):
            # 1 if patient eventually had this event after status_time
            event_after_status = int(ev == s["tokens"][-1] and s["times"][-1] > last_time)
            true_events.append(event_after_status)
            # time from status_time to event/censoring
            true_times.append(s["times"][-1] - last_time)
    
    true_events = np.array(true_events)
    true_times = np.array(true_times)
    risk_scores_flat = risk_scores.flatten()
    
    # 6. Compute Harrell's C-index per event
    c_index_dict = {}
    for k, ev in enumerate(event_types):
        mask = np.arange(len(risk_scores_flat)) % len(event_types) == k
        c_index = concordance_index(
            event_times=true_times[mask],
            predicted_scores=risk_scores_flat[mask],
            event_observed=true_events[mask]
        )
        c_index_dict[ev] = c_index
    
    return c_index_dict, risk_scores



In [ ]:
df_long_test["status_time"] = 5

c_index_dict, risk_scores = compute_dynamic_cindex(
    model=model,
    df_test=df_long_test,
    vocab=vocab,
    status_time_col="status_time",
    device="cpu"
)
print("Harrell's C-index per event:", c_index_dict)


In [ ]:
df_long_test.head(20)